# Lab: Polynomial Regression with scikit-learn

In this lab you'll learn how to fit a **Polynomial Regression** model using scikit-learn.

You already know the theory: polynomial regression creates new features ($x^2$, $x_1 x_2$, etc.) from the originals so that a standard `LinearRegression` model can fit curves. The key new ingredient at the code level is **`PolynomialFeatures`** — a scikit-learn transformer that generates those features for you automatically.

We'll continue with the familiar **Advertising** dataset (`tv`, `radio`, `newspaper` → `sales`). You already know from the theory file that there is a non-linear, curved relationship between advertising spend and sales. This lab focuses on *how to express that with sklearn*.

**What this lab teaches:**
1. How `PolynomialFeatures` transforms your feature matrix.
2. How to chain it with `LinearRegression` using a `Pipeline`.
3. How to use the fitted transformer to make predictions on new data.
4. How to visualise the fitted curve vs. a linear fit — so you can see the difference.

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# Plotting configuration
plt.style.use("fivethirtyeight")
sns.set_theme(style="white", palette="colorblind")
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["figure.dpi"] = 150

## 2. Load the Data

In [ ]:
sales_df = pd.read_csv("../data/raw/advertising.csv", usecols=["TV", "Radio", "Newspaper", "Sales"])
sales_df.columns = [col.lower() for col in sales_df.columns]
sales_df.head()

In [ ]:
sales_df.info()

## 3. Visualise the Raw Relationship

Before fitting any model, let's look at the scatter plot of `tv` spend vs `sales`. This gives you a feel for whether the relationship is linear or curved.

In [ ]:
plt.figure()
plt.scatter(sales_df["tv"], sales_df["sales"], alpha=0.6, color="steelblue", edgecolors="white", linewidths=0.5)
plt.title("TV Advertising Spend vs. Sales", fontweight="bold")
plt.xlabel("TV Advertising Spend (in thousands of dollars)")
plt.ylabel("Sales (in thousands of units)")
plt.tight_layout()
plt.show()

The scatter plot suggests a **curved (concave) relationship**: sales rise steeply at low TV spend and then start to level off at high spend (diminishing returns). A straight line would systematically under-predict at the extremes and over-predict in the middle — exactly the motivation for polynomial regression.

---

## 4. How `PolynomialFeatures` Works

`PolynomialFeatures(degree=2)` takes your original features and adds new ones:

| Original features | New features added by degree=2 |
|---|---|
| `tv` | `tv²` |
| `tv`, `radio` | `tv²`, `tv×radio`, `radio²` |
| `tv`, `radio`, `newspaper` | `tv²`, `tv×radio`, `tv×newspaper`, `radio²`, `radio×newspaper`, `newspaper²` |

The cross-terms (e.g., `tv×radio`) capture **interaction effects** — the idea that the combined effect of TV *and* Radio advertising is bigger than the sum of their individual effects.

Let's see this in action with a simple example.

In [ ]:
# Quick demo: what does PolynomialFeatures actually produce?
demo_X = sales_df[["tv", "radio"]].head(3)
print("Original features:")
print(demo_X.to_string(index=False))
print()

poly_demo = PolynomialFeatures(degree=2, include_bias=False)
demo_X_poly = poly_demo.fit_transform(demo_X)

print("After PolynomialFeatures(degree=2):")
print(pd.DataFrame(demo_X_poly, columns=poly_demo.get_feature_names_out()).to_string(index=False))

Notice that `get_feature_names_out()` shows you exactly which column is which — very useful for understanding what the model has access to.

---

## 5. Fit the Model using a Pipeline

Instead of manually calling `fit_transform` then `fit`, scikit-learn's **`Pipeline`** lets you chain a transformer and a model together into a single object. This is the idiomatic sklearn way:

- `pipeline.fit(X, y)` runs `PolynomialFeatures.fit_transform(X)` then `LinearRegression.fit(X_poly, y)` in one call.
- `pipeline.predict(X_new)` automatically transforms new data with the *same* fitted transformer before predicting — no risk of forgetting to transform.

We'll fit two models:
- **Linear baseline** — plain `LinearRegression` on `tv` alone.
- **Polynomial model** — `PolynomialFeatures(degree=2)` + `LinearRegression` on `tv` alone.

In [ ]:
X = sales_df[["tv"]]   # single feature so we can visualise the curve easily
y = sales_df["sales"]

# Linear baseline
linear_model = LinearRegression()
linear_model.fit(X, y)

# Polynomial model (degree 2) via Pipeline
poly_pipeline = Pipeline([
    ("poly_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("linear_regression", LinearRegression())
])
poly_pipeline.fit(X, y)

print("Both models fitted successfully.")

## 6. Inspect the Learned Parameters

In a pipeline you can access the named steps via `pipeline.named_steps`.

In [ ]:
lr = poly_pipeline.named_steps["linear_regression"]
pf = poly_pipeline.named_steps["poly_features"]

print(f"Intercept (β₀): {lr.intercept_:.4f}")
print()
for feature_name, coef in zip(pf.get_feature_names_out(), lr.coef_):
    print(f"  {feature_name:<10}  coef = {coef:+.6f}")

The negative coefficient on `tv^2` is what creates the concave curve — sales increase quickly at first but slow down as TV spend grows.

---

## 7. Visualise: Linear vs. Polynomial Fit

The best way to see what the polynomial model learned is to plot both fits over the data.

In [ ]:
tv_range = pd.DataFrame({"tv": np.linspace(X["tv"].min(), X["tv"].max(), 300)})

y_pred_linear = linear_model.predict(tv_range)
y_pred_poly   = poly_pipeline.predict(tv_range)

plt.figure()
plt.scatter(sales_df["tv"], sales_df["sales"], alpha=0.5, color="steelblue",
            edgecolors="white", linewidths=0.4, label="Observed data")
plt.plot(tv_range["tv"], y_pred_linear, color="tomato",  linewidth=2.5, label="Linear fit")
plt.plot(tv_range["tv"], y_pred_poly,   color="darkgreen", linewidth=2.5, label="Polynomial fit (degree=2)")

plt.title("Linear vs. Polynomial Regression on TV Spend", fontweight="bold")
plt.xlabel("TV Advertising Spend (in thousands of dollars)")
plt.ylabel("Sales (in thousands of units)")
plt.legend()
plt.tight_layout()
plt.show()

The polynomial curve follows the data much more closely — especially at the low and high ends of TV spend where the linear model misses.

---

## 8. Extend to Multiple Features

Now let's use both `tv` and `radio`. Adding a second feature means `PolynomialFeatures` also creates an **interaction term** (`tv × radio`) which captures the synergy effect: the combined impact of running TV *and* Radio ads simultaneously.

In [ ]:
X_multi = sales_df[["tv", "radio"]]

poly_pipeline_multi = Pipeline([
    ("poly_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("linear_regression", LinearRegression())
])
poly_pipeline_multi.fit(X_multi, y)

# Show the generated feature names
pf_multi = poly_pipeline_multi.named_steps["poly_features"]
lr_multi = poly_pipeline_multi.named_steps["linear_regression"]

print("Features and their coefficients:")
print(f"  Intercept            coef = {lr_multi.intercept_:+.4f}")
for name, coef in zip(pf_multi.get_feature_names_out(), lr_multi.coef_):
    print(f"  {name:<20}  coef = {coef:+.4f}")

The large positive coefficient on `tv radio` (the interaction term) confirms the synergy effect: markets that invest in *both* TV and Radio advertising at the same time see disproportionately higher sales.

---

## 9. Make Predictions on New Data

Because we used a `Pipeline`, predicting new data is just one call — the polynomial transformation is applied automatically.

In [ ]:
new_markets = pd.DataFrame({
    "tv":    [50.0, 150.0, 250.0],
    "radio": [10.0,  25.0,  40.0]
})

# Pipeline handles the PolynomialFeatures transform internally
predictions = poly_pipeline_multi.predict(new_markets)

new_markets["predicted_sales"] = predictions.round(2)
print(new_markets.to_string(index=False))

## 10. Summary

| | Linear Regression | Polynomial Regression |
|---|---|---|
| **Feature matrix** | Original features `X` | Expanded features `X, X², X×Y, …` |
| **Model fitted on** | `X` | `X_poly` |
| **Fitted curve** | Straight line / hyperplane | Curved surface |
| **sklearn class** | `LinearRegression` | `PolynomialFeatures` + `LinearRegression` |
| **Idiomatic packaging** | Direct | `Pipeline([poly, lr])` |
| **Predict on new data** | `model.predict(X_new)` | `pipeline.predict(X_new)` (transform included) |

### Key takeaways
- **`PolynomialFeatures`** is a transformer, not a model. It just creates new columns. The actual learning is still done by `LinearRegression`.
- **`Pipeline`** is the clean way to pair them: you fit and predict in one call, and the transformer is always applied consistently.
- **`get_feature_names_out()`** tells you exactly what the expanded feature matrix contains.
- The **interaction term** (`tv × radio`) captures synergy — something a purely linear model cannot express.

---

**Next:** [Polynomial Regression Implementation](./05_polynomial_regression_implementation.md)